# 🧩 GraphRAG with Neo4j's own libraries

This notebook uses **`neo4j-graphrag`** — Neo4j's first-party Python package — to do as much of
the work as the library covers, and hand-written Cypher only where it genuinely does not.

```
   documents  ──►  SimpleKGPipeline  ──►  (:Document)-[:FROM_DOCUMENT]-(:Chunk)
   (Postgres,      chunk · embed ·         (:__Entity__)-[:FROM_CHUNK]->(:Chunk)
    files, or      LLM extract ·           (:__Entity__)-[:YOUR_PREDICATE]->(:__Entity__)
    strings)       entity resolution ·
                   write                          │
                                                  ▼
                    HybridCypherRetriever  ──►  GraphRAG  ──►  answer
                    (official retrievers)      (official)
```

## Who does what

| Step | Who | Why |
|---|---|---|
| Chunking | **library** — `FixedSizeSplitter` | one parameter, maintained upstream |
| Embedding | **library** — `VertexAIEmbeddings` | swap provider by changing one class |
| **LLM entity + relation extraction** | **library** — `SimpleKGPipeline` | schema-guided, batched, with retries |
| Entity resolution | **library** — `perform_entity_resolution=True` | merges duplicates as it writes |
| Writing the graph | **library** | `:Document` / `:Chunk` / `:__Entity__` |
| Index creation | **library** — `create_vector_index`, `create_fulltext_index` | |
| Retrieval | **library** — Vector / Hybrid / HybridCypher / Text2Cypher | |
| Answer generation | **library** — `GraphRAG` | |
| Graph traversal, paths, blast radius, cycles | **§7, hand-written Cypher** | not in the package |
| Centrality, Leiden communities | **§7–§8** — `graphdatascience`, or a pure-Python fallback | not in the package |
| Corpus-wide global search | **§8, hand-written** | not in the package |

Roughly: **the library owns everything up to and including retrieval; §7 and §8 are the graph
reasoning it does not attempt.**

## Two things worth knowing before you start

**1. The pipeline is async.** `SimpleKGPipeline.run_async()` is a coroutine. §0.6 provides a
`run_sync()` helper that works both in Jupyter (where a loop is already running) and in a plain
script.

**2. It uses its own label conventions**, and they differ from a hand-rolled schema:
`:__Entity__` for entities, `:Chunk` for text, and **your declared predicates become real
relationship types** (`-[:SUPPLIES]->`, not `-[:REL {predicate}]->`). §4 discovers what was
actually written and configures §7 accordingly, so a version change in the library does not
silently break the traversal.

## Run order

**§0 → §3** ingests everything. **§4** verifies. **§5 → §6** answers questions with the
official retrievers. **§7** adds the graph queries the library omits. **§8** adds global search
— including a **pure-Python clustering fallback so it works on AuraDB Free**, which has no GDS.

---
# Section 0 — Setup

## 0.1 — Install

`neo4j-graphrag` ships provider extras. `[google]` pulls the Vertex AI LLM and embedder
classes; swap for `[openai]`, `[anthropic]`, `[cohere]`, `[mistralai]`, `[ollama]` as needed —
the rest of this notebook does not change.

`graphdatascience` is optional and only used in §7.3 / §8.1. **Skip it on AuraDB Free**, which
has no GDS; §8 has a pure-Python fallback that covers that case.

```bash
# self-hosted Neo4j 5.13+ (vector index), with GDS if you want centrality and Leiden
sudo docker run -d --name neo4j --restart unless-stopped \
  -p 7687:7687 -p 7474:7474 \
  -e NEO4J_AUTH=neo4j/CHANGE_ME_STRONG_PASSWORD \
  -e NEO4J_server_memory_heap_max__size=4G \
  -e NEO4J_PLUGINS='["graph-data-science"]' \
  -v /var/lib/neo4j/data:/data \
  neo4j:5-community
```

In [ ]:
# %pip install -q "neo4j-graphrag[google]" neo4j psycopg2-binary pandas
# %pip install -q graphdatascience          # optional: §7.3 and §8.1 (not on AuraDB Free)
# %pip install -q nest_asyncio              # only if your Jupyter complains about event loops

print("Installed. Restart the kernel if pip asked you to.")

## 0.2 — Configuration

Both passwords come from the environment:

```bash
export PGPASSWORD='...'          # only if you read documents from Postgres
export NEO4J_PASSWORD='...'
```

**`KG_SCHEMA` is the most important setting in this notebook.** It is what the extraction LLM
is allowed to produce. Open-ended extraction generates a long tail of junk types that fragments
the graph and inflates the bill; a declared schema is the single biggest quality lever you
have. `patterns` further constrains *which* node types may be joined by *which* relationship —
leave it empty while exploring, tighten it once you know your domain.

In [ ]:
import os

# ============================ Neo4j ============================
NEO4J_URI      = os.environ.get("NEO4J_URI", "bolt://127.0.0.1:7687")   # neo4j+s:// for Aura
NEO4J_USER     = os.environ.get("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.environ.get("NEO4J_PASSWORD")
NEO4J_DATABASE = os.environ.get("NEO4J_DATABASE", "neo4j")

# ============================ Vertex AI ============================
PROJECT_ID   = os.environ.get("GCP_PROJECT", "your-project-id")
LOCATION     = "us-central1"
EXTRACT_MODEL = "gemini-2.5-flash"      # SimpleKGPipeline's extraction LLM
CHAT_MODEL    = "gemini-2.5-flash"      # GraphRAG's answer LLM
SUMMARY_MODEL = "gemini-2.5-flash"      # community reports (§8)
EMBED_MODEL   = "text-embedding-005"    # Vertex embedding model
EMBED_DIM     = 768                     # ⬅️ MUST match the model's output dimensionality

# ============================ documents in ============================
DOC_SOURCE = "postgres"       # "postgres" | "files" | "none"
DOC_DIR    = "./documents"    # used when DOC_SOURCE == "files"
DOC_GLOB   = "*.txt"

PG_CONFIG = {
    "host":     os.environ.get("PGHOST", "127.0.0.1"),
    "port":     int(os.environ.get("PGPORT", 5432)),
    "dbname":   os.environ.get("PGDATABASE", "postgres"),
    "user":     os.environ.get("PGUSER", "postgres"),
    "password": os.environ.get("PGPASSWORD"),
    "connect_timeout": 10,
}
DOC_SOURCE_SQL = "SELECT doc_id, title, content FROM rag_documents"

# ============================ chunking ============================
CHUNK_SIZE    = 1500     # characters, not tokens — FixedSizeSplitter works in characters
CHUNK_OVERLAP = 200

# ============================ 🔑 the extraction schema ============================
# What the LLM is ALLOWED to produce. Replace these with your domain's vocabulary.
KG_SCHEMA = {
    "node_types": [
        "Organization", "Person", "Product", "Contract", "Certification",
        "Facility", "Location", "Document", "Event",
    ],
    "relationship_types": [
        "SUPPLIES", "SUBSIDIARY_OF", "PARENT_OF", "HOLDS_CERTIFICATION",
        "LOCATED_AT", "OPERATES", "PARTY_TO", "RESPONDS_TO", "AWARDED_TO",
        "REQUIRES", "EMPLOYED_BY", "MENTIONS",
    ],
    # Leave patterns empty to allow any declared relationship between any declared types.
    # Tighten it once you know your domain — it is the strongest quality lever after
    # node_types, because it stops the model inventing implausible joins.
    "patterns": [
        # ("Organization", "SUPPLIES", "Organization"),
        # ("Organization", "HOLDS_CERTIFICATION", "Certification"),
        # ("Organization", "SUBSIDIARY_OF", "Organization"),
    ],
}

PERFORM_ENTITY_RESOLUTION = True   # the library merges duplicates as it writes
ON_EXTRACTION_ERROR = "IGNORE"     # "IGNORE" keeps going; "RAISE" stops on the first failure
INGEST_CONCURRENCY = 3             # documents processed in parallel

# ============================ index names ============================
IDX_CHUNK_VEC = "chunk_embedding"
IDX_CHUNK_FTS = "chunk_fulltext"
IDX_COMM_VEC  = "community_embedding"

# ============================ traversal (§7) ============================
GRAPH_HOPS             = 2
GRAPH_HUB_DEGREE_MAX   = 60
GRAPH_MAX_NODES        = 60
GRAPH_MAX_EXTRA_CHUNKS = 12
GRAPH_MAX_FACTS        = 40
TOP_K                  = 5

# ============================ global search (§8) ============================
COMMUNITY_MIN_SIZE   = 3
COMMUNITY_MAX_NODES  = 60
COMMUNITY_MAX_EDGES  = 80
COMMUNITY_MAX_CHUNKS = 4
SUMMARY_MAX_CALLS    = None
SUMMARY_WORKERS      = 4
GLOBAL_PRESELECT_K   = 12
GLOBAL_BATCH_SIZE    = 5
GLOBAL_MIN_RATING    = 3
GLOBAL_TOP_POINTS    = 20
REFUSAL = "I don't have enough information in the indexed documents to answer that."

print(f"Neo4j     : {NEO4J_URI} db={NEO4J_DATABASE}")
print(f"Vertex AI : {PROJECT_ID}/{LOCATION}  extract={EXTRACT_MODEL}  "
      f"embed={EMBED_MODEL}@{EMBED_DIM}d")
print(f"Documents : {DOC_SOURCE}")
print(f"Schema    : {len(KG_SCHEMA['node_types'])} node types, "
      f"{len(KG_SCHEMA['relationship_types'])} relationship types, "
      f"{len(KG_SCHEMA['patterns'])} pattern(s)")
if not NEO4J_PASSWORD:
    print("\n⚠️  NEO4J_PASSWORD is not set — export it and re-run this cell.")

## 0.3 — Driver, LLM and embedder — all library classes

`VertexAILLM` and `VertexAIEmbeddings` come from the package. **Swapping provider is a
one-line change** — `OpenAILLM` / `OpenAIEmbeddings`, `AnthropicLLM`, `CohereLLM`,
`MistralAILLM`, `OllamaLLM` — and nothing downstream cares.

Note the `response_format` in `model_params`: extraction returns JSON, and telling the model so
removes a whole class of parse failures.

In [ ]:
import json, re, time, math, hashlib, asyncio, contextlib
from concurrent.futures import ThreadPoolExecutor

from neo4j import GraphDatabase
from neo4j_graphrag.llm import VertexAILLM
from neo4j_graphrag.embeddings import VertexAIEmbeddings
from neo4j_graphrag.indexes import create_vector_index, create_fulltext_index
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.experimental.components.text_splitters.fixed_size_splitter import (
    FixedSizeSplitter,
)
from neo4j_graphrag.retrievers import (
    VectorRetriever, VectorCypherRetriever,
    HybridRetriever, HybridCypherRetriever, Text2CypherRetriever,
)
from neo4j_graphrag.generation import GraphRAG

NEO4J_READY = False
driver = None


def neo4j_connect(verbose=True):
    """Open the driver and check the server is new enough for vector indexes."""
    global NEO4J_READY, driver
    if not NEO4J_PASSWORD:
        print("❌ NEO4J_PASSWORD is empty — export it and re-run §0.2.")
        return False
    try:
        driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD),
                                      max_connection_lifetime=3600, connection_timeout=20)
        driver.verify_connectivity()
        NEO4J_READY = True
        recs = driver.execute_query(
            "CALL dbms.components() YIELD name, versions, edition "
            "RETURN name, versions[0] AS version, edition",
            database_=NEO4J_DATABASE).records
        if recs and verbose:
            r = recs[0]
            print(f"✅ Neo4j connected: {r['name']} {r['version']} ({r['edition']})")
            if tuple(int(x) for x in str(r["version"]).split(".")[:2]) < (5, 13):
                print("⚠️  vector indexes need 5.13+ — §1 will fail on this server")
        return True
    except Exception as e:
        print(f"❌ Neo4j connection failed: {type(e).__name__}: {str(e)[:220]}")
        print("   Check the server is up, the port is reachable, the password is right,")
        print("   and the scheme matches (bolt:// self-hosted, neo4j+s:// for Aura).")
        return False


def cy(query, **params):
    """Plain Cypher, list of dicts. Used by §7 and §8 where the library has no equivalent."""
    recs = driver.execute_query(query, parameters_=params, database_=NEO4J_DATABASE).records
    return [r.data() for r in recs]


def cy_write(query, **params):
    return driver.execute_query(query, parameters_=params,
                                database_=NEO4J_DATABASE).summary.counters


def _need_neo4j(fn):
    if not NEO4J_READY:
        print(f"{fn}() needs a live Neo4j connection — run §0.3.")
        return False
    return True


neo4j_connect()

# ---- the library's LLM and embedder ----
llm = VertexAILLM(
    model_name=EXTRACT_MODEL,
    model_params={"temperature": 0, "response_mime_type": "application/json"},
)
answer_llm = VertexAILLM(
    model_name=CHAT_MODEL,
    model_params={"temperature": 0, "max_output_tokens": 2000},
)
embedder = VertexAIEmbeddings(model=EMBED_MODEL)

print(f"✅ library LLM + embedder ready ({EXTRACT_MODEL} / {EMBED_MODEL})")
print("   swap provider by changing the two classes above — nothing downstream changes")

## 0.4 — Read documents

Three sources, all producing the same shape: `[{doc_id, title, text}]`. Postgres is read-only
here — this notebook never writes back to it.

`SimpleKGPipeline` can also read PDFs itself with `from_file=True` and `run_async(file_path=…)`.
This notebook passes plain text instead, so the same code path serves database rows, text files
and strings.

In [ ]:
def read_documents_from_postgres(sql=None):
    """Read documents out of Postgres. SELECT only — nothing here writes."""
    import psycopg2
    import psycopg2.extras
    conn = psycopg2.connect(**PG_CONFIG)
    try:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(sql or DOC_SOURCE_SQL)
            rows = [dict(r) for r in cur.fetchall()]
    finally:
        conn.close()
    out = []
    for r in rows:
        text = r.get("content") or r.get("text") or ""
        if text.strip():
            out.append({"doc_id": str(r["doc_id"]),
                        "title": str(r.get("title") or r["doc_id"]),
                        "text": text})
    return out


def read_documents_from_files(directory=None, pattern=None):
    """Every matching file in a directory becomes one document."""
    from pathlib import Path
    p = Path(directory or DOC_DIR)
    if not p.exists():
        print(f"{p} does not exist.")
        return []
    out = []
    for f in sorted(p.glob(pattern or DOC_GLOB)):
        try:
            text = f.read_text(encoding="utf-8", errors="ignore")
        except Exception as e:
            print(f"  skip {f.name}: {type(e).__name__}")
            continue
        if text.strip():
            out.append({"doc_id": f.name, "title": f.stem, "text": text})
    return out


def load_documents():
    if DOC_SOURCE == "postgres":
        docs = read_documents_from_postgres()
    elif DOC_SOURCE == "files":
        docs = read_documents_from_files()
    else:
        docs = []
        print('DOC_SOURCE is "none" — assign DOCUMENTS yourself:')
        print('   DOCUMENTS = [{"doc_id": "D1", "title": "…", "text": "…"}]')
    if docs:
        chars = sum(len(d["text"]) for d in docs)
        print(f"✅ {len(docs)} document(s), {chars:,} characters "
              f"(~{chars // max(CHUNK_SIZE - CHUNK_OVERLAP, 1):,} chunks at the current settings)")
        for d in docs[:5]:
            print(f"   {d['doc_id']:<28} {len(d['text']):>8,} chars   {d['title'][:40]}")
        if len(docs) > 5:
            print(f"   … and {len(docs) - 5} more")
    return docs


try:
    DOCUMENTS = load_documents()
except Exception as _e:
    DOCUMENTS = []
    print(f"⚠️  could not read documents: {type(_e).__name__}: {str(_e)[:200]}")

## 0.5 — Running the async pipeline from a notebook

`SimpleKGPipeline.run_async()` is a coroutine, and `asyncio.run()` raises inside Jupyter
because a loop is already running. `run_sync()` handles both cases: it uses the running loop
via `nest_asyncio` when there is one, and `asyncio.run()` when there is not.

If `nest_asyncio` is not installed and you are in Jupyter, the cell tells you — you can also
just `await` the coroutine directly in a notebook cell.

In [ ]:
def run_sync(coro):
    """Run a coroutine from a notebook cell or a plain script."""
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)                 # no loop: plain script
    try:
        import nest_asyncio
        nest_asyncio.apply()
        return loop.run_until_complete(coro)     # Jupyter with nest_asyncio
    except ImportError:
        raise RuntimeError(
            "An event loop is already running (you are in Jupyter) and nest_asyncio is not "
            "installed.\n  Either:  %pip install nest_asyncio\n"
            "  or:      await the coroutine directly in a cell instead of calling run_sync().")


print("✅ run_sync() ready")

---
# Section 1 — Indexes, with the library's helpers

`create_vector_index` and `create_fulltext_index` come from `neo4j_graphrag.indexes`. They wrap
the Cypher DDL so you do not hand-write `OPTIONS {indexConfig: {...}}` and get the backtick
quoting wrong.

**`dimensions` must match your embedding model.** It is fixed at index creation — changing
`EMBED_DIM` later means dropping the index and re-embedding everything. `text-embedding-005`
emits 768; `gemini-embedding-001` emits 3072 natively and supports truncation. If §3 writes
vectors of a different width than the index expects, retrieval returns nothing with no error.

The vector index is created on the label `SimpleKGPipeline` writes (`Chunk`), against the
property it writes (`embedding`).

In [ ]:
CHUNK_LABEL     = "Chunk"          # what SimpleKGPipeline writes
CHUNK_TEXT_PROP = "text"
CHUNK_VEC_PROP  = "embedding"


def create_indexes(verbose=True):
    """Vector + full-text indexes via the library's helpers, plus a uniqueness constraint."""
    if not _need_neo4j("create_indexes"):
        return False

    create_vector_index(
        driver,
        IDX_CHUNK_VEC,
        label=CHUNK_LABEL,
        embedding_property=CHUNK_VEC_PROP,
        dimensions=int(EMBED_DIM),
        similarity_fn="cosine",
        neo4j_database=NEO4J_DATABASE,
        fail_if_exists=False,
    )
    create_fulltext_index(
        driver,
        IDX_CHUNK_FTS,
        label=CHUNK_LABEL,
        node_properties=[CHUNK_TEXT_PROP],
        neo4j_database=NEO4J_DATABASE,
        fail_if_exists=False,
    )
    # §8 stores community reports as nodes and pre-selects them by embedding, so they need
    # an index of their own. Same helper, different label.
    create_vector_index(
        driver,
        IDX_COMM_VEC,
        label="Community",
        embedding_property="embedding",
        dimensions=int(EMBED_DIM),
        similarity_fn="cosine",
        neo4j_database=NEO4J_DATABASE,
        fail_if_exists=False,
    )
    cy_write("CREATE CONSTRAINT community_key IF NOT EXISTS "
             "FOR (k:Community) REQUIRE k.key IS UNIQUE")

    try:
        cy("CALL db.awaitIndexes(300)")     # a still-building index returns nothing, not an error
    except Exception:
        pass

    if verbose:
        print("✅ indexes ready\n")
        for r in cy("SHOW INDEXES YIELD name, type, entityType, labelsOrTypes, properties, "
                    "state RETURN name, type, labelsOrTypes, properties, state ORDER BY name"):
            lbl = ",".join(r["labelsOrTypes"] or [])
            props = ",".join(r["properties"] or [])
            print(f"   {r['name']:<24} {r['type']:<10} {lbl:<12} {props:<16} {r['state']}")
    return True


def drop_vector_indexes():
    """Needed if you change EMBED_MODEL or EMBED_DIM — dimensionality is fixed at creation."""
    if not _need_neo4j("drop_vector_indexes"):
        return
    for name in (IDX_CHUNK_VEC, IDX_COMM_VEC):
        cy_write(f"DROP INDEX {name} IF EXISTS")
        print(f"   dropped {name}")
    print("Re-run create_indexes(), then re-ingest — old vectors are meaningless now.")


create_indexes()

---
# Section 2 — Preview what extraction will cost

💰 **§3 is the expensive step**: `SimpleKGPipeline` makes at least one LLM call per chunk to
extract entities and relationships, plus one embedding call per chunk.

This cell estimates the volume before you spend anything. If the number looks wrong, the levers
are `CHUNK_SIZE` (bigger chunks, fewer calls, coarser extraction) and running §3 on a subset
first.

In [ ]:
def estimate_ingestion(docs=None, verbose=True):
    """How many chunks, and therefore how many extraction and embedding calls."""
    docs = DOCUMENTS if docs is None else docs
    if not docs:
        print("No documents loaded — see §0.4.")
        return {"documents": 0, "chunks": 0}
    stride = max(CHUNK_SIZE - CHUNK_OVERLAP, 1)
    per_doc = [max(1, -(-len(d["text"]) // stride)) for d in docs]   # ceil division
    total = sum(per_doc)
    print(f"{len(docs)} document(s) → ~{total:,} chunk(s) at "
          f"CHUNK_SIZE={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}")
    print(f"   ≈ {total:,} extraction call(s) to {EXTRACT_MODEL}")
    print(f"   ≈ {total:,} embedding call(s) to {EMBED_MODEL}")
    if verbose:
        biggest = sorted(zip(docs, per_doc), key=lambda x: -x[1])[:5]
        print("   largest documents:")
        for d, n in biggest:
            print(f"     {d['doc_id']:<28} {len(d['text']):>8,} chars → ~{n:>4} chunks")
    if total > 2000:
        print(f"\n   ⚠️  {total:,} chunks is a real bill. Run §3 with limit=20 first and")
        print("      check the extraction quality in §4 before doing the whole corpus.")
    return {"documents": len(docs), "chunks": total}


estimate_ingestion()

---
# Section 3 — 🔑 Build the knowledge graph with `SimpleKGPipeline`

**This one class does five things**: splits each document into chunks, embeds them, calls the
LLM to extract entities and relationships constrained by `KG_SCHEMA`, resolves duplicate
entities, and writes the whole lexical + domain graph to Neo4j.

What lands in the database:

```
(:Document {path})
(:Chunk {text, index, embedding})-[:FROM_DOCUMENT]->(:Document)
(:Chunk)-[:NEXT_CHUNK]->(:Chunk)
(:__Entity__ {name, ...})-[:FROM_CHUNK]->(:Chunk)
(:__Entity__)-[:SUPPLIES|HOLDS_CERTIFICATION|…]->(:__Entity__)
```

Note the last line: **your declared relationship types become real Neo4j relationship types**,
not a `predicate` property. That is more idiomatic and lets Cypher filter in the pattern — and
it is why §4 discovers what was written instead of assuming.

**Start with `limit=20`.** Extraction quality depends almost entirely on `KG_SCHEMA`, and it is
much cheaper to discover that your schema is wrong after twenty documents than after all of
them.

In [ ]:
def build_kg_pipeline():
    """Construct the pipeline. Rebuilt each run so config edits take effect."""
    return SimpleKGPipeline(
        llm=llm,
        driver=driver,
        embedder=embedder,
        schema=KG_SCHEMA,                                  # constrains what the LLM may emit
        from_file=False,                                   # we pass text, not file paths
        text_splitter=FixedSizeSplitter(
            chunk_size=CHUNK_SIZE,
            chunk_overlap=CHUNK_OVERLAP,
            approximate=True,                              # respect word boundaries
        ),
        perform_entity_resolution=PERFORM_ENTITY_RESOLUTION,
        on_error=ON_EXTRACTION_ERROR,
        neo4j_database=NEO4J_DATABASE,
    )


async def _ingest_async(docs, concurrency):
    """Run the pipeline over every document, a few at a time."""
    kg = build_kg_pipeline()
    sem = asyncio.Semaphore(concurrency)
    done, failed = [], []

    async def _one(d):
        async with sem:
            try:
                await kg.run_async(text=d["text"])
                done.append(d["doc_id"])
                print(f"   ✓ {d['doc_id']}")
            except Exception as e:
                failed.append((d["doc_id"], f"{type(e).__name__}: {str(e)[:140]}"))
                print(f"   ✗ {d['doc_id']}: {type(e).__name__}: {str(e)[:120]}")

    await asyncio.gather(*(_one(d) for d in docs))
    return done, failed


def ingest(docs=None, limit=None, concurrency=None, dry_run=False):
    """💰 Chunk, embed, extract and write. Start with limit=20."""
    if not _need_neo4j("ingest"):
        return None
    docs = DOCUMENTS if docs is None else docs
    if not docs:
        print("No documents loaded — see §0.4.")
        return None
    if limit:
        docs = docs[:int(limit)]

    est = estimate_ingestion(docs, verbose=False)
    if dry_run:
        print("\nDRY RUN — nothing sent. Call ingest() without dry_run=True to proceed.")
        return None

    print(f"\nExtracting with schema: {len(KG_SCHEMA['node_types'])} node types, "
          f"{len(KG_SCHEMA['relationship_types'])} relationship types")
    t0 = time.perf_counter()
    done, failed = run_sync(_ingest_async(docs, concurrency or INGEST_CONCURRENCY))
    dt = time.perf_counter() - t0

    print(f"\n✅ {len(done)}/{len(docs)} document(s) in {dt:.0f}s")
    if failed:
        print(f"⚠️  {len(failed)} failed:")
        for doc_id, err in failed[:10]:
            print(f"     {doc_id}: {err}")
        print("   ON_EXTRACTION_ERROR='RAISE' in §0.2 stops on the first one if you want")
        print("   the traceback instead of a summary.")
    try:
        cy("CALL db.awaitIndexes(300)")
    except Exception:
        pass
    return {"ok": done, "failed": failed, "estimated_chunks": est["chunks"]}


# ⬇️ Start here. Drop dry_run and raise the limit once §4 says the extraction looks right.
ingest(limit=20, dry_run=True)

---
# Section 4 — Verify what the library actually wrote

**Do not skip this.** Two reasons:

1. **Extraction quality is the whole ballgame.** If the entity types look wrong here, fix
   `KG_SCHEMA` and re-run §3 on the same twenty documents — that is far cheaper than
   discovering it after the full corpus.
2. **Label conventions drift between library versions.** This cell discovers the labels and
   relationship types actually present and configures §7's traversal accordingly, so an
   upgrade does not silently break the graph queries.

`LEXICAL_RELS` is the important output: those are the relationships that hold the *text*
structure together (`FROM_CHUNK`, `NEXT_CHUNK`, `FROM_DOCUMENT`). §7 must exclude them from
traversal — otherwise two unrelated entities appear connected simply because they were
mentioned in the same chunk.

In [ ]:
ENTITY_LABEL = "__Entity__"
LEXICAL_RELS = ["FROM_CHUNK", "NEXT_CHUNK", "FROM_DOCUMENT", "PART_OF_DOCUMENT",
                "HAS_ENTITY", "NEXT", "HAS_CHUNK"]


def discover_schema(verbose=True):
    """Report what is in the database and adapt §7's constants to it."""
    global ENTITY_LABEL, CHUNK_LABEL, CHUNK_TEXT_PROP
    if not _need_neo4j("discover_schema"):
        return None

    labels = [r["label"] for r in cy("CALL db.labels() YIELD label RETURN label ORDER BY label")]
    rels = [r["relationshipType"] for r in
            cy("CALL db.relationshipTypes() YIELD relationshipType "
               "RETURN relationshipType ORDER BY relationshipType")]

    # Adapt to whatever the library version actually used.
    for cand in ("__Entity__", "__KGBuilder__", "Entity"):
        if cand in labels:
            ENTITY_LABEL = cand
            break
    for cand in ("Chunk", "__Chunk__"):
        if cand in labels:
            CHUNK_LABEL = cand
            break

    counts = {}
    for lbl in labels:
        counts[lbl] = cy(f"MATCH (n:`{lbl}`) RETURN count(n) AS n")[0]["n"]

    domain_rels = [r for r in rels if r not in LEXICAL_RELS]
    lexical_present = [r for r in rels if r in LEXICAL_RELS]

    if verbose:
        print("NODE LABELS")
        for lbl, n in sorted(counts.items(), key=lambda kv: -kv[1]):
            mark = ""
            if lbl == ENTITY_LABEL:
                mark = "   ← entities (§7 traverses these)"
            elif lbl == CHUNK_LABEL:
                mark = "   ← chunks (the vector index)"
            print(f"   {lbl:<24} {n:>8,}{mark}")

        print(f"\nDOMAIN RELATIONSHIP TYPES ({len(domain_rels)}) — these are your graph")
        for r in domain_rels[:25]:
            n = cy(f"MATCH ()-[x:`{r}`]->() RETURN count(x) AS n")[0]["n"]
            declared = "" if r in KG_SCHEMA["relationship_types"] else "   ⚠️ not in KG_SCHEMA"
            print(f"   {r:<28} {n:>8,}{declared}")
        if len(domain_rels) > 25:
            print(f"   … and {len(domain_rels) - 25} more")

        print(f"\nLEXICAL RELATIONSHIPS ({len(lexical_present)}) — excluded from traversal")
        print(f"   {', '.join(lexical_present) or '(none)'}")

        print("\nENTITY TYPES (the sub-label on each entity)")
        for r in cy(f"MATCH (n:`{ENTITY_LABEL}`) UNWIND labels(n) AS l "
                    f"WITH l WHERE NOT l IN ['{ENTITY_LABEL}', '__KGBuilder__'] "
                    f"RETURN l AS label, count(*) AS n ORDER BY n DESC LIMIT 20"):
            declared = "" if r["label"] in KG_SCHEMA["node_types"] else "   ⚠️ not in KG_SCHEMA"
            print(f"   {r['label']:<24} {r['n']:>8,}{declared}")

        undeclared = [r for r in domain_rels if r not in KG_SCHEMA["relationship_types"]]
        if undeclared:
            print(f"\n⚠️  {len(undeclared)} relationship type(s) the schema did not declare: "
                  f"{undeclared[:6]}")
            print("   Either add them to KG_SCHEMA, or tighten it — undeclared types usually")
            print("   mean the model is inventing vocabulary.")

    return {"labels": counts, "domain_rels": domain_rels, "lexical": lexical_present,
            "entity_label": ENTITY_LABEL, "chunk_label": CHUNK_LABEL}


def sample_extraction(n=5):
    """Read what the LLM actually pulled out. The fastest way to judge KG_SCHEMA."""
    if not _need_neo4j("sample_extraction"):
        return None
    rows = cy(f"MATCH (e:`{ENTITY_LABEL}`)-[r]->(t:`{ENTITY_LABEL}`) "
              f"WHERE NOT type(r) IN $lex "
              f"RETURN e.name AS src, labels(e) AS src_labels, type(r) AS predicate, "
              f"       t.name AS dst, labels(t) AS dst_labels LIMIT $n",
              lex=LEXICAL_RELS, n=int(n))
    if not rows:
        print("No entity→entity relationships yet. Run §3, or check KG_SCHEMA.")
        return []
    print("Sample of extracted relationships:")
    for r in rows:
        st = [l for l in r["src_labels"] if l not in (ENTITY_LABEL, "__KGBuilder__")]
        dt = [l for l in r["dst_labels"] if l not in (ENTITY_LABEL, "__KGBuilder__")]
        print(f"   ({str(r['src'])[:28]}:{','.join(st) or '?'}) "
              f"-[{r['predicate']}]-> "
              f"({str(r['dst'])[:28]}:{','.join(dt) or '?'})")
    return rows


SCHEMA_REPORT = discover_schema()
print()
sample_extraction()

---
# Section 5 — Retrieval, with the library's retrievers

Five official retrievers, each a different answer to "which passages should the model see".

| Retriever | How it finds passages | Use when |
|---|---|---|
| `VectorRetriever` | vector index only | the question means the same as the passage |
| `HybridRetriever` | vector **+** full-text, fused by the library | exact tokens matter — ids, SKUs, `Net-45` |
| `VectorCypherRetriever` | vector hit, **then your Cypher** | you want the graph around the hit |
| `HybridCypherRetriever` | hybrid hit, **then your Cypher** | 🔑 both of the above — the default here |
| `Text2CypherRetriever` | LLM writes Cypher, no vectors | the answer is an aggregate, not a passage |

**`HybridCypherRetriever` is the one that makes this GraphRAG rather than RAG.** Its
`retrieval_query` runs after the index search, with `node` (the matched chunk) and `score` in
scope. That is the hook for graph expansion: from the matched chunk, walk to its entities, walk
one hop further, and hand the model both the passage and the surrounding relationships.

In [ ]:
# `node` = the matched Chunk, `score` = its similarity. Everything after that is ours.
#
# Two things this query does that plain retrieval cannot:
#   1. collects the ENTITIES the chunk mentions, and their 1-hop neighbours
#   2. renders the relationships between them as text the model can cite
# The lexical relationships are excluded, so entities are not "connected" merely by sharing
# a chunk.
GRAPH_EXPANSION_QUERY = f"""
WITH node AS chunk, score
OPTIONAL MATCH (e:`{ENTITY_LABEL}`)-[:FROM_CHUNK]->(chunk)
WITH chunk, score, collect(DISTINCT e) AS entities
UNWIND (CASE WHEN size(entities) = 0 THEN [null] ELSE entities END) AS e
OPTIONAL MATCH (e)-[r]-(nb:`{ENTITY_LABEL}`)
WHERE NOT type(r) IN {LEXICAL_RELS!r}
WITH chunk, score,
     collect(DISTINCT coalesce(e.name, e.id)) AS entity_names,
     collect(DISTINCT
       coalesce(e.name, e.id) + ' -[' + type(r) + ']- ' + coalesce(nb.name, nb.id)
     )[0..25] AS relationships
RETURN chunk.text AS text,
       score AS score,
       entity_names AS entities,
       relationships AS relationships
"""


def build_retrievers():
    """All five, so you can compare them on your own questions."""
    if not _need_neo4j("build_retrievers"):
        return {}
    r = {}
    r["vector"] = VectorRetriever(
        driver, IDX_CHUNK_VEC, embedder,
        return_properties=[CHUNK_TEXT_PROP],
        neo4j_database=NEO4J_DATABASE,
    )
    r["hybrid"] = HybridRetriever(
        driver, IDX_CHUNK_VEC, IDX_CHUNK_FTS, embedder,
        return_properties=[CHUNK_TEXT_PROP],
        neo4j_database=NEO4J_DATABASE,
    )
    r["vector_graph"] = VectorCypherRetriever(
        driver, IDX_CHUNK_VEC,
        retrieval_query=GRAPH_EXPANSION_QUERY,
        embedder=embedder,
        neo4j_database=NEO4J_DATABASE,
    )
    r["hybrid_graph"] = HybridCypherRetriever(          # 🔑 the default
        driver, IDX_CHUNK_VEC, IDX_CHUNK_FTS,
        retrieval_query=GRAPH_EXPANSION_QUERY,
        embedder=embedder,
        neo4j_database=NEO4J_DATABASE,
    )
    try:
        r["text2cypher"] = Text2CypherRetriever(
            driver=driver, llm=answer_llm, neo4j_database=NEO4J_DATABASE,
        )
    except Exception as e:
        print(f"   (Text2CypherRetriever unavailable: {type(e).__name__})")
    return r


RETRIEVERS = build_retrievers()
print(f"✅ {len(RETRIEVERS)} retriever(s): {', '.join(RETRIEVERS)}")
print("   hybrid_graph is the GraphRAG one — vector + keyword, then graph expansion")


def compare_retrievers(question, top_k=3, which=("vector", "hybrid", "hybrid_graph")):
    """Run one question through several retrievers. This is how you justify the choice."""
    for name in which:
        rt = RETRIEVERS.get(name)
        if rt is None:
            continue
        print(f"\n{'─' * 74}\n{name}\n{'─' * 74}")
        try:
            res = rt.search(query_text=question, top_k=top_k)
        except Exception as e:
            print(f"   ✗ {type(e).__name__}: {str(e)[:160]}")
            continue
        for i, item in enumerate(res.items, 1):
            body = str(item.content)
            print(f"  {i}. {body[:260]}{'…' if len(body) > 260 else ''}")
    return None

---
# Section 6 — Answering, with the library's `GraphRAG`

`GraphRAG(retriever, llm).search(query_text=…)` is the whole thing: it retrieves, builds the
prompt, calls the model and returns the answer. `return_context=True` gives you the retrieved
items back so you can see *why* it answered that way — always use it while you are still
building trust in the pipeline.

`ask_text2cypher()` is a different shape and worth keeping separate. It does not retrieve
passages at all: the LLM writes a Cypher query, Neo4j runs it, and the rows are the answer.
That is the right tool for *"how many suppliers hold an expired certification?"* — an
aggregate, which no amount of passage retrieval will compute correctly.

In [ ]:
def build_rag(retriever_name="hybrid_graph"):
    rt = RETRIEVERS.get(retriever_name)
    if rt is None:
        print(f"No retriever named '{retriever_name}'. Available: {list(RETRIEVERS)}")
        return None
    return GraphRAG(retriever=rt, llm=answer_llm)


RAG = build_rag("hybrid_graph")


def ask(question, retriever_name="hybrid_graph", top_k=None, show_context=True,
        verbose=True):
    """Ask a question through the library's GraphRAG pipeline."""
    rag = RAG if retriever_name == "hybrid_graph" else build_rag(retriever_name)
    if rag is None:
        return None
    t0 = time.perf_counter()
    try:
        res = rag.search(
            query_text=question,
            retriever_config={"top_k": int(top_k or TOP_K)},
            return_context=True,
            response_fallback=REFUSAL,
        )
    except Exception as e:
        print(f"✗ {type(e).__name__}: {str(e)[:220]}")
        return None
    dt = (time.perf_counter() - t0) * 1000

    if verbose:
        print(res.answer)
        if show_context and getattr(res, "retriever_result", None):
            print(f"\nRetrieved by '{retriever_name}' "
                  f"({len(res.retriever_result.items)} item(s), {dt:.0f} ms):")
            for i, item in enumerate(res.retriever_result.items, 1):
                body = str(item.content).replace("\n", " ")
                print(f"  [{i}] {body[:200]}{'…' if len(body) > 200 else ''}")
    return res


def ask_text2cypher(question, verbose=True):
    """The LLM writes Cypher and Neo4j answers. For aggregates, not passages."""
    rt = RETRIEVERS.get("text2cypher")
    if rt is None:
        print("Text2CypherRetriever is not available in this install.")
        return None
    try:
        res = rt.search(query_text=question)
    except Exception as e:
        print(f"✗ {type(e).__name__}: {str(e)[:220]}")
        print("   Text-to-Cypher fails loudly on schemas it cannot map. That is better than")
        print("   a confident wrong number — rephrase using your real labels.")
        return None
    if verbose:
        print(f"Generated Cypher:\n   {getattr(res, 'metadata', {}).get('cypher', '(hidden)')}\n")
        for i, item in enumerate(res.items, 1):
            print(f"  {i}. {str(item.content)[:220]}")
    return res


print("✅ ask(question) · ask(question, 'vector') · ask_text2cypher(question)")
print("   compare_retrievers(question) to see the difference on your own data")

---
# Section 7 — What the library does not do

`neo4j-graphrag` stops at retrieval. Everything below is hand-written Cypher over the graph
`SimpleKGPipeline` built, and it is the reason to use a graph database rather than a vector
store.

**All of it excludes `LEXICAL_RELS` and requires every node on a path to be an entity.**
Without that, two unrelated companies look connected because a chunk happened to mention both.

## 7.1 — Degree, and the hub guard

`degree` counts **entity-to-entity relationships only**. It is cached because the hub guard
reads it on every hop, and it must be **recomputed after every ingest** — a stale degree
silently changes which nodes the walk refuses to relay through.

The guard itself: an entity above `GRAPH_HUB_DEGREE_MAX` can be an **answer** but is never a
**relay**. Without it, two hops through your own organisation reaches the entire graph, and
"everything" is not a retrieval result.

In [ ]:
def refresh_degree(verbose=True):
    """Cache entity degree, counting only entity→entity relationships."""
    if not _need_neo4j("refresh_degree"):
        return 0
    cy_write(f"""
        MATCH (n:`{ENTITY_LABEL}`)
        OPTIONAL MATCH (n)-[r]-(m:`{ENTITY_LABEL}`)
        WHERE NOT type(r) IN $lex
        WITH n, count(r) AS d
        SET n.degree = d
    """, lex=LEXICAL_RELS)
    r = cy(f"MATCH (n:`{ENTITY_LABEL}`) "
           f"RETURN count(n) AS n, max(n.degree) AS mx, "
           f"       count(CASE WHEN n.degree > $h THEN 1 END) AS hubs",
           h=int(GRAPH_HUB_DEGREE_MAX))[0]
    if verbose:
        print(f"✅ degree cached on {r['n']:,} entities (max {r['mx']})")
        print(f"   {r['hubs']} above the hub guard — reachable as answers, never as relays")
        for row in cy(f"MATCH (n:`{ENTITY_LABEL}`) WHERE n.degree > $h "
                      f"RETURN coalesce(n.name, n.id) AS label, n.degree AS d "
                      f"ORDER BY d DESC LIMIT 5", h=int(GRAPH_HUB_DEGREE_MAX)):
            print(f"     {row['d']:>5}  {str(row['label'])[:44]}")
    return r["n"]


def find_entity(text, limit=15, show=True):
    """Find a starting point by name."""
    if not _need_neo4j("find_entity"):
        return []
    rows = cy(f"MATCH (n:`{ENTITY_LABEL}`) "
              f"WHERE toLower(coalesce(n.name, n.id, '')) CONTAINS toLower($t) "
              f"RETURN elementId(n) AS eid, coalesce(n.name, n.id) AS label, "
              f"       [l IN labels(n) WHERE NOT l IN ['{ENTITY_LABEL}','__KGBuilder__']] AS types, "
              f"       coalesce(n.degree, 0) AS degree "
              f"ORDER BY size(coalesce(n.name, n.id, '')) ASC LIMIT $k",
              t=text, k=int(limit))
    if show:
        for r in rows:
            print(f"  {','.join(r['types']) or '?':<20} {str(r['label'])[:44]:<44} "
                  f"(deg {r['degree']})")
    return rows


def _resolve(hint):
    rows = find_entity(hint, limit=10, show=False)
    return (rows[0]["label"], rows[0]["eid"]) if rows else None


def hub_ranking(limit=15, show=True):
    """Highest-degree entities — single points of failure. No plugin needed."""
    if not _need_neo4j("hub_ranking"):
        return []
    rows = cy(f"MATCH (n:`{ENTITY_LABEL}`) "
              f"RETURN coalesce(n.name, n.id) AS label, coalesce(n.degree,0) AS degree, "
              f"       [l IN labels(n) WHERE NOT l IN ['{ENTITY_LABEL}','__KGBuilder__']] AS types "
              f"ORDER BY degree DESC LIMIT $k", k=int(limit))
    if show:
        print("Most connected entities (degree = concentration risk):")
        for r in rows:
            flag = "  ← above hub guard" if r["degree"] > GRAPH_HUB_DEGREE_MAX else ""
            print(f"  {r['degree']:>4}  {','.join(r['types']) or '?':<18} "
                  f"{str(r['label'])[:40]}{flag}")
    return rows


refresh_degree()

## 7.2 — 🔑 Paths, blast radius, cycles, shared dependencies

The queries a vector store cannot answer at all. Each takes entity **names** and resolves them,
so you can call them with the words a person would use.

| Function | Question |
|---|---|
| `path_between()` | *How is A connected to B?* — the **route**, not just reachability |
| `all_paths_between()` | *In how many different ways?* |
| `paths_via()` | *Connected by ownership, or only by trading?* |
| `blast_radius()` | *What breaks if this fails?* |
| `shared_dependencies()` | *What do these three all rely on?* |
| `find_cycles()` | *Any circular dependencies?* |

In [ ]:
_PATH_RETURN = (
    "RETURN length(p) AS hops, "
    "       [x IN nodes(p) | coalesce(x.name, x.id)] AS labels, "
    "       [r IN relationships(p) | type(r)]        AS predicates "
)


def _render_path(labels, predicates, indent="   "):
    for i, p in enumerate(predicates):
        print(f"{indent}{str(labels[i])[:32]:<32} --[{p}]-->  {str(labels[i+1])[:32]}")


def path_between(a_hint, b_hint, max_hops=6, show=True):
    """Shortest path between two entities, by name."""
    if not _need_neo4j("path_between"):
        return None
    a, b = _resolve(a_hint), _resolve(b_hint)
    if not a or not b:
        print(f'Could not resolve "{a_hint if not a else b_hint}" — try find_entity(...).')
        return None
    if show:
        print(f'  "{a_hint}" → {a[0]}    "{b_hint}" → {b[0]}')
    rows = cy(f"MATCH (a:`{ENTITY_LABEL}`), (b:`{ENTITY_LABEL}`) "
              f"WHERE elementId(a) = $a AND elementId(b) = $b "
              f"MATCH p = shortestPath((a)-[*..%d]-(b)) "
              f"WHERE ALL(x IN nodes(p) WHERE x:`{ENTITY_LABEL}`) "
              f"  AND ALL(r IN relationships(p) WHERE NOT type(r) IN $lex) "
              % int(max_hops) + _PATH_RETURN,
              a=a[1], b=b[1], lex=LEXICAL_RELS)
    if not rows:
        print(f"No path within {max_hops} hops.")
        return None
    r = rows[0]
    if show:
        print(f"{r['hops']} hop(s):")
        _render_path(r["labels"], r["predicates"])
    return r


def all_paths_between(a_hint, b_hint, max_hops=6, limit=10, show=True):
    """EVERY shortest path — 'connected three different ways' is itself the answer."""
    if not _need_neo4j("all_paths_between"):
        return None
    a, b = _resolve(a_hint), _resolve(b_hint)
    if not a or not b:
        print("Could not resolve one of the endpoints.")
        return None
    rows = cy(f"MATCH (a:`{ENTITY_LABEL}`), (b:`{ENTITY_LABEL}`) "
              f"WHERE elementId(a) = $a AND elementId(b) = $b "
              f"MATCH p = allShortestPaths((a)-[*..%d]-(b)) "
              f"WHERE ALL(x IN nodes(p) WHERE x:`{ENTITY_LABEL}`) "
              f"  AND ALL(r IN relationships(p) WHERE NOT type(r) IN $lex) "
              % int(max_hops) + _PATH_RETURN + "LIMIT $k",
              a=a[1], b=b[1], lex=LEXICAL_RELS, k=int(limit))
    if show:
        print(f"{len(rows)} distinct shortest path(s):")
        for i, r in enumerate(rows, 1):
            print(f"  path {i} ({r['hops']} hops):")
            _render_path(r["labels"], r["predicates"], indent="     ")
    return rows


def paths_via(a_hint, b_hint, predicates, max_hops=6, limit=10, show=True):
    """Paths using ONLY the given relationship types."""
    if not _need_neo4j("paths_via"):
        return None
    a, b = _resolve(a_hint), _resolve(b_hint)
    if not a or not b:
        print("Could not resolve one of the endpoints.")
        return None
    rows = cy(f"MATCH (a:`{ENTITY_LABEL}`), (b:`{ENTITY_LABEL}`) "
              f"WHERE elementId(a) = $a AND elementId(b) = $b "
              f"MATCH p = (a)-[rs*1..%d]-(b) "
              f"WHERE ALL(x IN nodes(p) WHERE x:`{ENTITY_LABEL}`) "
              f"  AND ALL(r IN rs WHERE type(r) IN $preds) "
              % int(max_hops) + _PATH_RETURN + "ORDER BY hops ASC LIMIT $k",
              a=a[1], b=b[1], preds=list(predicates), k=int(limit))
    if show:
        if not rows:
            print(f"No path using only {list(predicates)}.")
        for i, r in enumerate(rows, 1):
            print(f"  path {i} ({r['hops']} hops):")
            _render_path(r["labels"], r["predicates"], indent="     ")
    return rows


def blast_radius(hint, hops=2, limit=60, show=True):
    """What is downstream of this entity — 'what breaks if it fails?'"""
    if not _need_neo4j("blast_radius"):
        return []
    ent = _resolve(hint)
    if not ent:
        print(f'Could not resolve "{hint}" — try find_entity(...).')
        return []
    rows = cy(f"MATCH (a:`{ENTITY_LABEL}`) WHERE elementId(a) = $id "
              f"MATCH p = (a)-[*1..%d]-(b:`{ENTITY_LABEL}`) "
              f"WHERE ALL(x IN nodes(p) WHERE x:`{ENTITY_LABEL}`) "
              f"  AND ALL(r IN relationships(p) WHERE NOT type(r) IN $lex) "
              f"  AND ALL(x IN nodes(p)[0..-1] WHERE coalesce(x.degree, 0) <= $hub) "
              f"WITH b, min(length(p)) AS hop "
              f"RETURN elementId(b) AS eid, coalesce(b.name, b.id) AS label, "
              f"  [l IN labels(b) WHERE NOT l IN ['{ENTITY_LABEL}','__KGBuilder__']] AS types, "
              f"  coalesce(b.degree,0) AS degree, hop "
              f"ORDER BY hop ASC, degree DESC LIMIT $k" % int(hops),
              id=ent[1], lex=LEXICAL_RELS, hub=int(GRAPH_HUB_DEGREE_MAX), k=int(limit))
    if show:
        by_type = {}
        for r in rows:
            by_type.setdefault(",".join(r["types"]) or "?", []).append(r)
        print(f"Blast radius of {ent[0]} ({len(rows)} entities within {hops} hops):")
        for t, rs in sorted(by_type.items(), key=lambda kv: -len(kv[1])):
            names = ", ".join(f"{str(r['label'])[:22]}({r['hop']}h)" for r in rs[:5])
            more = f" +{len(rs) - 5} more" if len(rs) > 5 else ""
            print(f"  {t:<20} {len(rs):>3}  {names}{more}")
    return rows


def shared_dependencies(hints, min_shared=2, limit=25, show=True):
    """What do these entities have IN COMMON? The concentration-risk query."""
    if not _need_neo4j("shared_dependencies"):
        return []
    ids = [e[1] for e in (_resolve(h) for h in hints) if e]
    if len(ids) < 2:
        print("Need at least two resolvable entities.")
        return []
    rows = cy(f"MATCH (a:`{ENTITY_LABEL}`) WHERE elementId(a) IN $ids "
              f"MATCH (a)-[r]-(c:`{ENTITY_LABEL}`) "
              f"WHERE NOT type(r) IN $lex AND NOT elementId(c) IN $ids "
              f"WITH c, collect(DISTINCT coalesce(a.name,a.id)) AS via, "
              f"     collect(DISTINCT type(r)) AS preds "
              f"WHERE size(via) >= $min "
              f"RETURN coalesce(c.name, c.id) AS label, "
              f"  [l IN labels(c) WHERE NOT l IN ['{ENTITY_LABEL}','__KGBuilder__']] AS types, "
              f"  via, size(via) AS shared, preds "
              f"ORDER BY shared DESC LIMIT $k",
              ids=ids, lex=LEXICAL_RELS, min=int(min_shared), k=int(limit))
    if show:
        print(f"Shared by >= {min_shared} of {len(ids)} entities:")
        for r in rows:
            print(f"  {r['shared']}x  {','.join(r['types']) or '?':<18} "
                  f"{str(r['label'])[:34]:<34} via {r['preds'][:3]}")
    return rows


def find_cycles(max_len=4, limit=15, show=True):
    """Directed cycles — circular dependencies, ownership loops."""
    if not _need_neo4j("find_cycles"):
        return []
    rows = cy(f"MATCH p = (a:`{ENTITY_LABEL}`)-[*2..%d]->(a) "
              f"WHERE ALL(x IN nodes(p) WHERE x:`{ENTITY_LABEL}`) "
              f"  AND ALL(r IN relationships(p) WHERE NOT type(r) IN $lex) "
              % int(max_len) + _PATH_RETURN + "ORDER BY hops ASC LIMIT $k",
              lex=LEXICAL_RELS, k=int(limit))
    if show:
        print(f"{len(rows)} cycle(s) of length <= {max_len}:")
        for r in rows:
            print("  (%d) %s" % (r["hops"], " -> ".join(str(x)[:18] for x in r["labels"])))
    return rows


print("✅ path_between() · all_paths_between() · paths_via()")
print("   blast_radius() · shared_dependencies() · find_cycles() · hub_ranking()")

## 7.3 — Answering from a path or a blast radius

`ask()` in §6 lets retrieval choose the passages. **These let the graph choose them**: resolve
the entities, find the route or the radius, then pull the chunks those entities came from.

Retrieval never sees the question's second entity, and would never surface the documents that
explain the middle of a chain — which is exactly what makes these worth having.

In [ ]:
def _chunks_for_entities(labels, limit=None):
    """Chunks the given entities were extracted from, via [:FROM_CHUNK]."""
    return cy(f"MATCH (e:`{ENTITY_LABEL}`)-[:FROM_CHUNK]->(c:`{CHUNK_LABEL}`) "
              f"WHERE coalesce(e.name, e.id) IN $names "
              f"WITH c, count(DISTINCT e) AS hits "
              f"RETURN c.{CHUNK_TEXT_PROP} AS text, hits "
              f"ORDER BY hits DESC LIMIT $k",
              names=list(labels), k=int(limit or GRAPH_MAX_EXTRA_CHUNKS))


def _answer_from_graph(question, context_block, chunks):
    """One grounded generation from graph facts plus the passages behind them."""
    sources = "\n\n".join(f"[{i}] {c['text']}" for i, c in enumerate(chunks, 1)) or "(none)"
    prompt = (
        "You are an analyst assistant. Answer using ONLY the GRAPH CONTEXT and the numbered "
        "SOURCES below.\n\n"
        "Rules:\n"
        "1. Lead with the connection or the impact itself, in order, naming each link.\n"
        "2. Cite supporting detail with its source number, e.g. [2]. Cite the graph as "
        "[graph].\n"
        "3. The GRAPH CONTEXT is verified — every link is a stated relationship, not an "
        "inference.\n"
        "4. Quote figures, dates and identifiers exactly as written.\n"
        "5. If a link is unexplained by the sources, say which one rather than inventing a "
        "reason.\n"
        f'6. If nothing here answers the question, reply exactly: "{REFUSAL}"\n'
        "7. Be concise.\n\n"
        f"GRAPH CONTEXT:\n{context_block}\n\n"
        f"SOURCES:\n{sources}\n\n"
        f"QUESTION: {question}")
    return answer_llm.invoke(prompt).content.strip()


def ask_path(question, a_hint, b_hint, max_hops=6):
    """'How is A connected to B?' — the graph finds the route, the route finds the text."""
    if not _need_neo4j("ask_path"):
        return None
    p = path_between(a_hint, b_hint, max_hops=max_hops, show=True)
    if not p:
        print("  → no path; falling back to the retriever")
        return ask(question)
    chain = "\n".join(f"  {p['labels'][i]} --[{pr}]--> {p['labels'][i+1]}"
                      for i, pr in enumerate(p["predicates"]))
    chunks = _chunks_for_entities(p["labels"])
    if not chunks:
        print("  ⚠️  path found but no chunk mentions any entity on it")
    answer = _answer_from_graph(
        question, f"GRAPH PATH ({p['hops']} hops):\n{chain}", chunks)
    print("\n" + answer)
    print(f"\n  {len(chunks)} passage(s), all chosen by the graph")
    return {"answer": answer, "path": p, "chunks": chunks}


def ask_impact(question, hint, hops=2):
    """'What breaks if X fails?' — blast radius supplies the entities."""
    if not _need_neo4j("ask_impact"):
        return None
    radius = blast_radius(hint, hops=hops, show=True)
    if not radius:
        return ask(question)
    names = [r["label"] for r in radius]
    ent = _resolve(hint)
    context = (f"IMPACT SCOPE: {len(radius)} entities within {hops} hops of "
               f"{ent[0] if ent else hint}:\n"
               + "\n".join(f"  {r['hop']}h  {','.join(r['types']) or '?'}  {r['label']}"
                           for r in radius[:30]))
    chunks = _chunks_for_entities(([ent[0]] if ent else []) + names)
    answer = _answer_from_graph(question, context, chunks)
    print("\n" + answer)
    print(f"\n  {len(chunks)} passage(s), all chosen by the graph")
    return {"answer": answer, "radius": radius, "chunks": chunks}


print("✅ ask_path(q, a, b) · ask_impact(q, entity)")

---
# Section 8 — Global search

Everything above answers questions about **specific things**. Now ask:

> *"What are the recurring compliance risks across all our suppliers?"*

**No chunk contains that answer.** It exists only as a pattern spread across the corpus, so it
has to be *synthesised* — once, at index time, not per question.

```
 INDEX (once)   1. CLUSTER    the entity graph into communities
                2. SUMMARISE  one LLM call per community → a written report
                3. EMBED      into the Community vector index from §1

 QUERY          4. PRESELECT  the K reports closest to the question
                5. MAP        each scores what it contributes
                6. REDUCE     the best points become one answer
```

**Clustering works without GDS.** §8.1 uses GDS Leiden when the plugin is present and falls
back to **label propagation implemented in Python** when it is not — which is what makes this
section work on **AuraDB Free**, where GDS is unavailable. The fallback is coarser than Leiden
and says so rather than pretending otherwise.

`member_hash` fingerprints each community's exact membership, so re-clustering after an ingest
keeps the summaries of communities that did not change. Without it, every rebuild silently
re-spends the whole summarisation budget.

## 8.1 — Cluster: GDS Leiden, or pure Python

In [ ]:
def gds_available():
    if not NEO4J_READY:
        return False
    try:
        cy("RETURN gds.version() AS v")
        return True
    except Exception:
        return False


def _fetch_entity_graph():
    """The entity-only adjacency, as plain Python. Used by the fallback clusterer."""
    rows = cy(f"MATCH (a:`{ENTITY_LABEL}`)-[r]-(b:`{ENTITY_LABEL}`) "
              f"WHERE NOT type(r) IN $lex "
              f"RETURN DISTINCT coalesce(a.name,a.id) AS a, coalesce(b.name,b.id) AS b",
              lex=LEXICAL_RELS)
    adj = {}
    for r in rows:
        if r["a"] is None or r["b"] is None or r["a"] == r["b"]:
            continue
        adj.setdefault(r["a"], set()).add(r["b"])
        adj.setdefault(r["b"], set()).add(r["a"])
    return adj


def _label_propagation(adj, rounds=20):
    """Deterministic label propagation. Coarser than Leiden, but needs no plugin.

    Each node repeatedly adopts the most common label among its neighbours, ties broken by
    the smallest label so the result is reproducible. Converges in a handful of rounds on
    graphs of this size.
    """
    labels = {n: n for n in adj}
    order = sorted(adj)
    for _ in range(rounds):
        changed = False
        for n in order:
            counts = {}
            for nb in adj[n]:
                counts[labels[nb]] = counts.get(labels[nb], 0) + 1
            if not counts:
                continue
            best = min(sorted(counts), key=lambda l: (-counts[l], l))
            if best != labels[n]:
                labels[n] = best
                changed = True
        if not changed:
            break
    groups = {}
    for n, l in labels.items():
        groups.setdefault(l, []).append(n)
    return [sorted(v) for v in groups.values()]


def _leiden_gds():
    """GDS Leiden, writing a `community` property. Returns groups or None."""
    try:
        cy("CALL gds.graph.drop('kg_graph', false) YIELD graphName")
    except Exception:
        pass
    try:
        cy(f"CALL gds.graph.project('kg_graph', '{ENTITY_LABEL}', "
           f"  '*', {{relationshipProperties: []}}) YIELD graphName")
        cy("CALL gds.leiden.write('kg_graph', {writeProperty: 'community', "
           "  randomSeed: 42, concurrency: 1}) YIELD communityCount")
    except Exception as e:
        print(f"  ⚠️  GDS Leiden failed ({type(e).__name__}: {str(e)[:120]})")
        return None
    rows = cy(f"MATCH (n:`{ENTITY_LABEL}`) WHERE n.community IS NOT NULL "
              f"RETURN n.community AS cid, "
              f"       collect(coalesce(n.name, n.id)) AS members")
    return [r["members"] for r in rows]


def _member_hash(names):
    return hashlib.sha256("|".join(sorted(names)).encode()).hexdigest()


def build_communities(verbose=True):
    """Cluster the entity graph and write (:Community) nodes. Preserves unchanged summaries."""
    if not _need_neo4j("build_communities"):
        return 0

    groups, algo = None, None
    if gds_available():
        groups = _leiden_gds()
        algo = "GDS Leiden"
    if groups is None:
        adj = _fetch_entity_graph()
        if not adj:
            print("No entity→entity relationships — nothing to cluster. Check §4.")
            return 0
        groups = _label_propagation(adj)
        algo = "label propagation (pure Python — no GDS on this server)"

    groups = [g for g in groups if len(g) >= COMMUNITY_MIN_SIZE]
    if not groups:
        print(f"No communities of at least {COMMUNITY_MIN_SIZE} members. "
              f"Try lowering COMMUNITY_MIN_SIZE.")
        return 0
    groups.sort(key=len, reverse=True)

    prior = {r["h"]: r for r in cy(
        "MATCH (k:Community) WHERE k.summary IS NOT NULL "
        "RETURN k.member_hash AS h, k.title AS title, k.summary AS summary, "
        "       k.rating AS rating, k.embedding AS embedding")}
    cy_write("MATCH (k:Community) DETACH DELETE k")

    rows, kept = [], 0
    for cid, members in enumerate(groups):
        h = _member_hash(members)
        old = prior.get(h)
        if old:
            kept += 1
        rows.append({"key": f"0:{cid}", "community_id": cid, "size": len(members),
                     "member_hash": h, "members": members,
                     "title": old["title"] if old else None,
                     "summary": old["summary"] if old else None,
                     "rating": old["rating"] if old else None,
                     "embedding": old["embedding"] if old else None})

    for i in range(0, len(rows), 50):
        cy_write(f"""
            UNWIND $rows AS row
            MERGE (k:Community {{key: row.key}})
            SET k.community_id = row.community_id, k.size = row.size,
                k.member_hash = row.member_hash, k.title = row.title,
                k.summary = row.summary, k.rating = row.rating,
                k.embedding = row.embedding
            WITH k, row
            UNWIND row.members AS nm
            MATCH (n:`{ENTITY_LABEL}`) WHERE coalesce(n.name, n.id) = nm
            MERGE (n)-[:IN_COMMUNITY]->(k)
        """, rows=rows[i:i + 50])

    print(f"✅ {len(groups)} communities ({algo})")
    if verbose:
        for i, m in enumerate(groups[:8]):
            print(f"   #{i:<3} {len(m):>4} members   e.g. {', '.join(m[:3])}")
    if kept:
        print(f"   {kept} existing summary(ies) preserved — membership unchanged")
    print(f"   {len(groups) - kept} need summarising → summarize_communities()")
    return len(groups)


build_communities()

## 8.2 — Write the community reports

💰 **One LLM call per unsummarised community.** Prints an estimate first and stops at
`SUMMARY_MAX_CALLS`.

Each prompt gets three kinds of context, and the third is what makes the difference: the
community's **entities**, the **relationships** between them, and a few **representative
passages** — the chunks those entities were extracted from. Without passages the model
summarises a list of names and produces something generic; with them it summarises the actual
documents and quotes real figures.

In [ ]:
COMMUNITY_PROMPT = """You are summarising one cluster of a business knowledge graph.

Write a report on this community: what it is about, who and what is in it, and what the
relationships between them mean for the business.

ENTITIES:
{entities}

RELATIONSHIPS:
{relationships}

REPRESENTATIVE PASSAGES:
{passages}

Return RAW JSON only:
{{"title": "<short specific name, 3-8 words>",
  "summary": "<200-350 words. Lead with what this community IS. Then the important entities
              and what connects them. Then anything notable: risks, dependencies, conflicts,
              obligations, gaps. Use ONLY the information above. Name names and quote figures
              exactly as written.>",
  "rating": <0-10, how important this community is for understanding the business overall>}}"""


def _parse_json_reply(text):
    text = (text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        a, b = text.find("{"), text.rfind("}")
        if a != -1 and b != -1:
            return json.loads(text[a:b + 1])
        raise


def _community_context(key):
    ents = cy(f"MATCH (n:`{ENTITY_LABEL}`)-[:IN_COMMUNITY]->(:Community {{key: $k}}) "
              f"RETURN coalesce(n.name, n.id) AS label, "
              f"  [l IN labels(n) WHERE NOT l IN ['{ENTITY_LABEL}','__KGBuilder__']] AS types "
              f"LIMIT $lim", k=key, lim=int(COMMUNITY_MAX_NODES))
    names = [e["label"] for e in ents]
    rels = cy(f"MATCH (a:`{ENTITY_LABEL}`)-[r]->(b:`{ENTITY_LABEL}`) "
              f"WHERE coalesce(a.name,a.id) IN $n AND coalesce(b.name,b.id) IN $n "
              f"  AND NOT type(r) IN $lex "
              f"RETURN coalesce(a.name,a.id) AS src, type(r) AS predicate, "
              f"       coalesce(b.name,b.id) AS dst LIMIT $lim",
              n=names, lex=LEXICAL_RELS, lim=int(COMMUNITY_MAX_EDGES))
    chunks = _chunks_for_entities(names, limit=COMMUNITY_MAX_CHUNKS)

    ent = "\n".join(f"  - {e['label']} ({','.join(e['types']) or '?'})"
                    for e in ents) or "  (none)"
    rel = "\n".join(f"  - {str(r['src'])[:40]} --[{r['predicate']}]--> {str(r['dst'])[:40]}"
                    for r in rels) or "  (none)"
    psg = "\n\n".join(f"  {c['text'][:900]}" for c in chunks) or "  (none)"
    return ent, rel, psg


def summarize_communities(only_missing=True, max_calls=None, dry_run=False, verbose=True):
    """One LLM call per community. Resumable, capped, safe to run twice."""
    if not _need_neo4j("summarize_communities"):
        return 0
    max_calls = SUMMARY_MAX_CALLS if max_calls is None else max_calls
    where = "k.summary IS NULL" if only_missing else "true"
    todo = cy(f"MATCH (k:Community) WHERE {where} "
              f"RETURN k.key AS key, k.size AS size ORDER BY k.size DESC")
    if max_calls:
        todo = todo[:int(max_calls)]
    total = cy("MATCH (k:Community) RETURN count(k) AS n")[0]["n"]

    print(f"{total} communities, {len(todo)} to summarise "
          f"→ {len(todo)} LLM call(s) + {len(todo)} embedding(s)")
    if dry_run:
        print("DRY RUN — nothing sent.")
        return 0
    if not todo:
        print("Nothing to do.")
        return 0

    summary_llm = VertexAILLM(model_name=SUMMARY_MODEL,
                              model_params={"temperature": 0,
                                            "response_mime_type": "application/json"})

    def _one(c):
        try:
            ent, rel, psg = _community_context(c["key"])
            data = _parse_json_reply(summary_llm.invoke(COMMUNITY_PROMPT.format(
                entities=ent, relationships=rel, passages=psg)).content)
            title = str(data.get("title", ""))[:300]
            summary = str(data.get("summary", ""))
            rating = float(data.get("rating") or 5)
            vec = embedder.embed_query(f"{title}\n{summary}")
            return {"key": c["key"], "title": title, "summary": summary,
                    "rating": rating, "embedding": vec}
        except Exception as e:
            print(f"  ⚠️  {c['key']}: {type(e).__name__}: {str(e)[:120]}")
            return None

    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=SUMMARY_WORKERS) as pool:
        out = [r for r in pool.map(_one, todo) if r]
    if out:
        for i in range(0, len(out), 25):
            cy_write("UNWIND $rows AS row MATCH (k:Community {key: row.key}) "
                     "SET k.title = row.title, k.summary = row.summary, "
                     "    k.rating = row.rating, k.embedding = row.embedding",
                     rows=out[i:i + 25])
        try:
            cy("CALL db.awaitIndexes(120)")
        except Exception:
            pass
    if verbose:
        for r in sorted(out, key=lambda x: -x["rating"])[:10]:
            print(f"   {r['key']:<8} rating {r['rating']:>4.1f}   {r['title']}")
    print(f"✅ Summarised {len(out)}/{len(todo)} in {time.perf_counter() - t0:.0f}s")
    return len(out)


summarize_communities(dry_run=True)

## 8.3 — Global search: preselect → map → reduce

**Pre-selection is what makes this affordable.** Naively, every question maps over every report:
200 communities is 40 LLM calls *per question*. Instead the question is embedded and matched
against `Community.embedding` through the vector index §1 created, and only the
`GLOBAL_PRESELECT_K` closest reports are mapped. **Cost stops growing with corpus size.**

**The map prompt is explicitly told that returning nothing is correct.** Without that, models
manufacture relevance for every report they see and the reduce step drowns in weak points.

In [ ]:
MAP_PROMPT = """You are answering a question using summaries of communities in a knowledge graph.

From the COMMUNITY REPORTS below, extract the points that help answer the QUESTION.

Score each point 0-100 for how directly and importantly it helps answer THIS question.
If a report is irrelevant, produce no point for it. An empty list is a correct and expected
answer when nothing is relevant — do not invent relevance.

Each point must be self-contained: name the entities, quote the figures. The writer of the
final answer will not see these reports, only your points.

Return RAW JSON only:
{{"points": [{{"description": "<specific, self-contained statement>", "score": 85}}]}}

QUESTION: {question}

COMMUNITY REPORTS:
{reports}"""

REDUCE_PROMPT = """You are a business analyst writing a final answer.

Below are key points gathered from across the entire document corpus, each with an importance
score. Synthesise them into one coherent answer to the QUESTION.

Rules:
- Use ONLY these points. Add nothing from outside them.
- Organise by theme, not by score. Lead with what matters most.
- Where points conflict, say so rather than silently picking one.
- Be specific: name names, quote figures exactly as given.
- If the points do not answer the question, reply exactly: "{refusal}"

QUESTION: {question}

KEY POINTS:
{points}"""


def _select_communities(question, k, min_rating):
    """The K most relevant reports via the Community vector index. Falls back to all."""
    if k:
        try:
            qv = embedder.embed_query(question)
            rows = cy("CALL db.index.vector.queryNodes($idx, $k, $qv) YIELD node, score "
                      "WITH node AS c, score "
                      "WHERE c.summary IS NOT NULL AND coalesce(c.rating, 0) >= $mr "
                      "RETURN c.community_id AS community_id, c.title AS title, "
                      "       c.summary AS summary, c.rating AS rating "
                      "ORDER BY score DESC",
                      idx=IDX_COMM_VEC, k=int(k) * 3, qv=qv, mr=float(min_rating))
            if rows:
                return rows[:int(k)], "preselected"
        except Exception as e:
            print(f"  ⚠️  preselection unavailable ({type(e).__name__}) — using all reports.")
    rows = cy("MATCH (k:Community) WHERE k.summary IS NOT NULL "
              "  AND coalesce(k.rating, 0) >= $mr "
              "RETURN k.community_id AS community_id, k.title AS title, "
              "       k.summary AS summary, k.rating AS rating "
              "ORDER BY k.rating DESC, k.size DESC", mr=float(min_rating))
    return rows, "all"


def global_search(question, preselect_k=..., min_rating=None, verbose=True):
    """Map-reduce over community reports. For questions about the corpus as a whole."""
    if not _need_neo4j("global_search"):
        return None
    k = GLOBAL_PRESELECT_K if preselect_k is ... else preselect_k
    min_rating = GLOBAL_MIN_RATING if min_rating is None else min_rating

    comms, how = _select_communities(question, k, min_rating)
    if not comms:
        msg = ("No community summaries yet. Run build_communities() then "
               "summarize_communities().")
        print(msg)
        return {"answer": msg, "points": [], "refused": True}

    batches = [comms[i:i + GLOBAL_BATCH_SIZE]
               for i in range(0, len(comms), GLOBAL_BATCH_SIZE)]
    if verbose:
        print(f"   map: {len(comms)} report(s) ({how}) in {len(batches)} batch(es)…")

    map_llm = VertexAILLM(model_name=SUMMARY_MODEL,
                          model_params={"temperature": 0,
                                        "response_mime_type": "application/json"})

    def _map(batch):
        reports = "\n\n".join(
            f"--- Community {c['community_id']}: {c['title']} (rating {c['rating']}) ---\n"
            f"{c['summary']}" for c in batch)
        try:
            data = _parse_json_reply(map_llm.invoke(
                MAP_PROMPT.format(question=question, reports=reports)).content)
            return [p for p in (data.get("points") or [])
                    if isinstance(p, dict) and p.get("description")]
        except Exception as e:
            print(f"  ⚠️  map batch failed: {type(e).__name__}: {str(e)[:110]}")
            return []

    with ThreadPoolExecutor(max_workers=SUMMARY_WORKERS) as pool:
        points = [p for b in pool.map(_map, batches) for p in b]

    def _score(p):
        try:
            return float(p.get("score") or 0)
        except (TypeError, ValueError):
            return 0.0

    points.sort(key=_score, reverse=True)
    top = points[:GLOBAL_TOP_POINTS]
    if verbose:
        print(f"   {len(points)} key point(s), keeping top {len(top)}")
    if not top:
        print(REFUSAL)
        return {"answer": REFUSAL, "points": [], "refused": True}

    answer = answer_llm.invoke(REDUCE_PROMPT.format(
        question=question, refusal=REFUSAL,
        points="\n".join(f"  - ({_score(p):.0f}) {p['description']}" for p in top))).content

    print("\n" + answer.strip())
    print(f"\nSynthesised from {len(comms)} report(s) ({how}); {len(top)} points used.")
    for p in top[:5]:
        print(f"  ({_score(p):.0f}) {str(p['description'])[:100]}")
    return {"answer": answer.strip(), "points": top, "communities_read": len(comms),
            "refused": False}


def communities(top=20):
    if not _need_neo4j("communities"):
        return []
    rows = cy("MATCH (k:Community) "
              "RETURN k.community_id AS cid, k.size AS size, k.rating AS rating, "
              "       k.title AS title, k.summary IS NOT NULL AS done "
              "ORDER BY k.rating DESC, k.size DESC LIMIT $t", t=int(top))
    for r in rows:
        flag = "" if r["done"] else "  ⚠️ not summarised"
        print(f"  #{r['cid']:<3} size={r['size']:<4} rating={r['rating']} "
              f"{(r['title'] or '(untitled)')[:46]}{flag}")
    return rows


print("✅ global_search(question) · communities()")

---
# Section 9 — Operating it

## 9.1 — Rebuild order, cost, backup

### When documents change

```python
DOCUMENTS = load_documents()      # free
ingest()                          # 💰 chunk + embed + LLM extract, per chunk
discover_schema()                 # free — did the schema hold?
refresh_degree()                  # free — MUST run after every ingest
build_communities()               # free — preserves unchanged summaries
summarize_communities()           # 💰 only communities that actually changed
```

**`SimpleKGPipeline` has no content-hash cache.** Re-running `ingest()` on the same document
extracts and embeds it again, at full price. Track which `doc_id`s you have already processed
yourself, or delete and rebuild deliberately:

```python
cy_write("MATCH (d:Document)<-[:FROM_DOCUMENT]-(c:Chunk) DETACH DELETE c, d")
```

### 🔴 Backup

Chunks, embeddings and the extracted graph all live only in Neo4j, and the embeddings and
extraction cost real money. **Neo4j Community has no online backup** — the dump requires
stopping the database:

```bash
sudo docker stop neo4j
sudo docker run --rm -v /var/lib/neo4j/data:/data -v $(pwd):/backup \
  neo4j:5-community neo4j-admin database dump neo4j --to-path=/backup
sudo docker start neo4j
```

On AuraDB you get one exportable snapshot at a time; on **AuraDB Free the instance is deleted
after 30 days of inactivity**.

### Where it goes wrong

| Symptom | Cause | Fix |
|---|---|---|
| Retrieval returns nothing | index still building, or dimension mismatch | `CALL db.awaitIndexes(300)`; check `EMBED_DIM` against the model |
| Entity types you never declared | schema too loose | tighten `KG_SCHEMA`, add `patterns`, re-run §3 |
| Everything extracted as one type | schema too tight, or wrong vocabulary | widen `node_types` to your domain's words |
| `§7` traversal returns half the graph | `LEXICAL_RELS` missing a type | re-run `discover_schema()` — it lists what exists |
| `expand`/paths return nothing | degree stale, or hub guard too tight | `refresh_degree()`; raise `GRAPH_HUB_DEGREE_MAX` |
| One giant community | label propagation merging through a hub | install GDS for Leiden, or raise `COMMUNITY_MIN_SIZE` |
| `gds.*` not found | no plugin | expected on AuraDB Free — §8.1 falls back automatically |
| Event loop error in §3 | Jupyter + `asyncio.run` | `%pip install nest_asyncio`, or `await` directly |

### Before production

- **Least privilege** — Postgres needs `SELECT` only; the query path needs a read-only Neo4j
  user, only ingestion writes.
- **Pin the library version.** `neo4j-graphrag` is young and `SimpleKGPipeline` lives under
  `experimental`. Pin it, and re-run §4 after any upgrade — that cell exists precisely to catch
  a convention change before it silently breaks §7.
- **Watch the split.** If nothing routes to `ask_path` / `ask_impact` / `global_search` after a
  month of real use, the graph is not earning its keep and plain `VectorRetriever` would do.

In [ ]:
def health():
    """One call: what is in the database, and what still needs doing."""
    if not _need_neo4j("health"):
        return None
    def n(q, **kw):
        try:
            return cy(q, **kw)[0]["n"]
        except Exception:
            return 0

    stats = {
        "documents": n("MATCH (d:Document) RETURN count(d) AS n"),
        "chunks": n(f"MATCH (c:`{CHUNK_LABEL}`) RETURN count(c) AS n"),
        "unembedded": n(f"MATCH (c:`{CHUNK_LABEL}`) WHERE c.{CHUNK_VEC_PROP} IS NULL "
                        f"RETURN count(c) AS n"),
        "entities": n(f"MATCH (e:`{ENTITY_LABEL}`) RETURN count(e) AS n"),
        "relationships": n(f"MATCH (:`{ENTITY_LABEL}`)-[r]-(:`{ENTITY_LABEL}`) "
                           f"WHERE NOT type(r) IN $lex RETURN count(DISTINCT r) AS n",
                           lex=LEXICAL_RELS),
        "no_degree": n(f"MATCH (e:`{ENTITY_LABEL}`) WHERE e.degree IS NULL "
                       f"RETURN count(e) AS n"),
        "communities": n("MATCH (k:Community) RETURN count(k) AS n"),
        "summarised": n("MATCH (k:Community) WHERE k.summary IS NOT NULL "
                        "RETURN count(k) AS n"),
    }
    print("CONTENTS")
    for k in ("documents", "chunks", "entities", "relationships", "communities",
              "summarised"):
        print(f"  {k:<16} {stats[k]:>9,}")

    print("\nINDEXES")
    try:
        for r in cy("SHOW INDEXES YIELD name, type, state, populationPercent "
                    "WHERE type IN ['VECTOR','FULLTEXT'] "
                    "RETURN name, type, state, populationPercent ORDER BY name"):
            ok = "✅" if r["state"] == "ONLINE" else "⚠️ "
            print(f"  {ok} {r['name']:<22} {r['type']:<10} {r['state']} "
                  f"({r['populationPercent']:.0f}%)")
    except Exception as e:
        print(f"  (could not read indexes: {type(e).__name__})")

    print("\nWARNINGS")
    warned = False
    for cond, msg in [
        (stats["unembedded"], f"{stats['unembedded']:,} chunk(s) with no embedding"),
        (stats["no_degree"], f"{stats['no_degree']:,} entity(ies) with no degree "
                             f"— run refresh_degree()"),
        (stats["entities"] and not stats["relationships"],
         "entities exist but no relationships — KG_SCHEMA may be too tight"),
        (stats["communities"] > stats["summarised"],
         f"{stats['communities'] - stats['summarised']} community(ies) unsummarised"),
    ]:
        if cond:
            print(f"  ⚠️  {msg}")
            warned = True
    if not warned:
        print("  none")

    print("\nREADINESS")
    for name, ok in [
        ("retrieval (§5-§6)", stats["chunks"] > 0 and stats["unembedded"] == 0),
        ("graph queries (§7)", stats["relationships"] > 0),
        ("global search (§8)", stats["summarised"] > 0),
        ("GDS Leiden", gds_available()),
    ]:
        note = ""
        if name == "GDS Leiden" and not ok:
            note = "   (fine — §8.1 falls back to label propagation)"
        print(f"  {'✅' if ok else '⚠️ '} {name}{note}")
    return stats


health()

## 9.2 — Try it

Replace the hints with entities from **your** graph — `find_entity("...")` lists them.

In [ ]:
if NEO4J_READY:
    print("─" * 74)
    print("What did the extraction actually produce?")
    print("─" * 74)
    sample_extraction(n=8)

    print("\n" + "─" * 74)
    print("Most connected entities")
    print("─" * 74)
    hub_ranking(limit=8)

    print("\n" + "─" * 74)
    print("Find a starting point")
    print("─" * 74)
    find_entity("a", limit=8)          # ⬅️ a real substring from your data
else:
    print("Neo4j not connected — run §0.3.")

In [ ]:
# ⬇️ uncomment once §0–§8 have run against your data

# --- the library's own pipeline ---
# compare_retrievers("What are the payment terms?")
# ask("What certifications does this supplier hold?")
# ask("What certifications does this supplier hold?", retriever_name="vector")

# --- text-to-Cypher: for aggregates, not passages ---
# ask_text2cypher("How many organizations hold a certification?")

# --- what the library does not do (§7) ---
# path_between("Tidewater", "Produce RFP")
# ask_path("How are these connected, and what does it mean for our exposure?",
#          "Tidewater", "Produce RFP")
# ask_impact("What breaks if this supplier is terminated?", "Tidewater")
# shared_dependencies(["Supplier A", "Supplier B", "Supplier C"])
# find_cycles()

# --- corpus-wide (§8) ---
# global_search("What are the recurring compliance risks across all suppliers?")

print("Uncomment a line above once the pipeline has run against your data.")